In [2]:
from pathlib import Path
import os

PROJECT_DIR = Path(r"C:\Users\ASUS\OneDrive\Documents\PROJEK ML GIAN & GANESH")

for folder in ("src", "models", "data"):
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

Working directory: C:\Users\ASUS\OneDrive\Documents\PROJEK ML GIAN & GANESH


In [4]:
import json
import joblib
import sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, confusion_matrix, classification_report

from src.features import clean_data, build_features, RAW_FEATURES

df_raw = pd.read_csv(PROJECT_DIR / "raw.csv").drop_duplicates().reset_index(drop=True)
print("Jumlah baris unik:", len(df_raw))

train_df, temp_df = train_test_split(
    df_raw, test_size=0.30, stratify=df_raw["cacat"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["cacat"], random_state=42
)

test_df.to_csv(PROJECT_DIR / "test.csv", index=False)

X_train_base = clean_data(train_df)[RAW_FEATURES]
y_train = train_df["cacat"]
X_val_base = clean_data(val_df)[RAW_FEATURES]
y_val = val_df["cacat"]

cat_cols = ["mesin", "shift", "bahan"]
num_cols_base = [c for c in RAW_FEATURES if c not in cat_cols]

prep_base = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols_base),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

baseline_pipe = Pipeline([
    ("prep", prep_base),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])
baseline_pipe.fit(X_train_base, y_train)
val_proba_base = baseline_pipe.predict_proba(X_val_base)[:, 1]
pr_auc_base = average_precision_score(y_val, val_proba_base)
print(f"[Validation] PR-AUC Baseline (Logistic Regression): {pr_auc_base:.4f}")


X_train_feat = build_features(train_df).drop(columns=["waktu_produksi", "cacat"])
X_val_feat = build_features(val_df).drop(columns=["waktu_produksi", "cacat"])
feature_names = list(X_train_feat.columns)
num_cols_feat = [c for c in feature_names if c not in cat_cols]

prep_final = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols_feat),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), cat_cols),
])

final_pipe = Pipeline([
    ("prep", prep_final),
    ("clf", HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.05, max_depth=6, random_state=42
    ))
])
final_pipe.fit(X_train_feat, y_train)
val_proba_final = final_pipe.predict_proba(X_val_feat)[:, 1]
pr_auc_final = average_precision_score(y_val, val_proba_final)
print(f"[Validation] PR-AUC Model Final (HistGradientBoosting): {pr_auc_final:.4f}")

best_t, min_cost = 0.5, float("inf")
for t in np.linspace(0.01, 0.80, 160):
    preds = (val_proba_final >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
    cost = fn * 500_000 + fp * 25_000
    if cost < min_cost:
        min_cost = cost
        best_t = float(t)

print(f"[Validation] Threshold Bisnis Optimal: {best_t:.4f} (Estimasi Rugi Val: Rp{min_cost:,.0f})")

X_test_feat = build_features(test_df).drop(columns=["waktu_produksi", "cacat"])
y_test = test_df["cacat"]
test_proba = final_pipe.predict_proba(X_test_feat)[:, 1]
test_pred = (test_proba >= best_t).astype(int)
pr_auc_test = average_precision_score(y_test, test_proba)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

print(f"\n Hasil Evaluasi pada Test Set:")
print(f"PR-AUC Test: {pr_auc_test:.4f}")
print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(classification_report(y_test, test_pred))

# 5. Simpan Artifact untuk Web App & GitHub
joblib.dump(final_pipe, "models/model.joblib")
metadata = {
    "model_type": "HistGradientBoostingClassifier",
    "sklearn_version": sklearn.__version__,
    "features": feature_names,
    "threshold": round(best_t, 4),
    "metrics": {
        "val_pr_auc_baseline": round(pr_auc_base, 4),
        "val_pr_auc_final": round(pr_auc_final, 4),
        "test_pr_auc_final": round(pr_auc_test, 4),
    },
    "business_cost_assumption": {"FN_idr": 500000, "FP_idr": 25000},
}
with open("models/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Berhasil menyimpan models/model.joblib dan models/metadata.json!")

Jumlah baris unik: 25000
[Validation] PR-AUC Baseline (Logistic Regression): 0.5140
[Validation] PR-AUC Model Final (HistGradientBoosting): 0.6720
[Validation] Threshold Bisnis Optimal: 0.0497 (Estimasi Rugi Val: Rp29,225,000)

 Hasil Evaluasi pada Test Set:
PR-AUC Test: 0.6669
Confusion Matrix: TN=2890, FP=569, FN=40, TP=251
              precision    recall  f1-score   support

           0       0.99      0.84      0.90      3459
           1       0.31      0.86      0.45       291

    accuracy                           0.84      3750
   macro avg       0.65      0.85      0.68      3750
weighted avg       0.93      0.84      0.87      3750

Berhasil menyimpan models/model.joblib dan models/metadata.json!
